# Probabilidad y Estadística — Proyecto final
## Clasificador de máxima verosimilitud para MNIST

Carga y exploración del conjunto de datos MNIST de dígitos escritos a mano.

## Derivación analítica — Clasificador de máxima verosimilitud

---

### 1. ¿Qué tengo?

Tengo un conjunto de datos de entrenamiento. Cada dato tiene dos partes:

$$(x_i,\, y_i)$$

donde $x_i$ es una imagen y $y_i$ es su clase real.

En MNIST, cada imagen es de $28 \times 28$ píxeles. Aunque la imagen se ve como una matriz,
para hacer álgebra la convierto en un vector:

$$x_i \in \mathbb{R}^{784} \qquad \text{pues } 28 \cdot 28 = 784$$

La etiqueta pertenece al conjunto:

$$y_i \in \{0, 1, 2, 3, 4, 5, 6, 7, 8, 9\}$$

Entonces el dataset es:

$$\mathcal{D} = \{(x_1, y_1),\, (x_2, y_2),\, \dots,\, (x_N, y_N)\}$$

donde $N = 60\,000$ y $N_c$ denota el número de imágenes de la clase $c$.

---

### 2. ¿Qué quiero construir?

Quiero una función que reciba una imagen y devuelva una clase:

$$f(x) = \hat{y}$$

Aquí $x$ es una imagen nueva, que el modelo no ha visto, y $\hat{y}$ es la clase predicha.
El sombrerito en $\hat{y}$ significa: *esta no necesariamente es la clase verdadera, es mi estimación.*

---

### 3. ¿Qué significa «decidir»?

Supongamos que para una imagen $x$, yo pudiera darle un puntaje a cada clase.
Por ejemplo: $S_0(x)=2.1$, $S_1(x)=0.3$, $S_2(x)=5.7$, $S_3(x)=1.4$, etc.

Si el puntaje más alto lo tiene la clase 2, entonces predigo $\hat{y}=2$.
Esta idea se escribe con $\arg\max$.

---

### 4. ¿Qué es $\arg\max$?

La expresión $\max_c\, S_c(x)$ te devuelve el **valor máximo**.
Pero la expresión $\arg\max_c\, S_c(x)$ te devuelve **quién produjo ese valor máximo**.

Con los puntajes del ejemplo anterior: $\max_c\, S_c(x) = 5.7$, pero $\arg\max_c\, S_c(x) = 2$,
porque la clase 2 fue la que produjo el puntaje más alto.

Para clasificar no queremos el valor del puntaje máximo. Queremos saber **qué clase gana**.
Por eso escribimos:

$$\hat{y} = \arg\max_c\, S_c(x)$$

---

### 5. ¿Qué puntaje deberíamos usar?

Como este es un proyecto de probabilidad, el puntaje más natural es:

$$P(y=c \mid x)$$

Esto se lee: *probabilidad de que la clase sea $c$, dado que observé la imagen $x$.*

Entonces, si tengo una imagen nueva, podría calcular $P(y=0 \mid x)$, $P(y=1 \mid x)$, …, $P(y=9 \mid x)$
y escoger la clase con mayor probabilidad:

$$\hat{y} = \arg\max_{c \in \{0,\dots,9\}} P(y=c \mid x)$$

Esta es la idea central de la clasificación probabilística.

---

### 6. Pero hay un problema

La cantidad que quiero es $P(y=c \mid x)$, pero directamente no sé calcularla.
No tengo una fórmula inmediata que me diga: *dada esta imagen, la probabilidad de que sea un 7 es 0.91.*

Necesito transformar esa expresión en términos más manejables. Ahí entra Bayes.

---

### 7. Bayes entra porque cambia la pregunta

El teorema de Bayes dice:

$$P(y=c \mid x) = \frac{P(y=c)\;p(x \mid y=c)}{p(x)}$$

Originalmente preguntábamos: *dada la imagen, ¿qué tan probable es la clase?*
Bayes lo reescribe usando $p(x \mid y=c)$: *si yo supiera que la clase es $c$, ¿qué tan probable
o compatible sería observar una imagen como $x$?*

Eso es más fácil de modelar, porque podemos mirar todas las imágenes de una misma clase
y estudiar cómo se distribuyen.

---

### 8. Interpretación de cada término de Bayes

**Primer término:** $P(y=c)$ — la **probabilidad previa** (prior) de la clase $c$.
Responde: *antes de mirar la imagen, ¿qué tan común es la clase $c$?*
Por ejemplo, si en el dataset hay muchos más unos que nueves, entonces $P(y=1) > P(y=9)$.
En MNIST normalmente las clases están más o menos balanceadas, pero igual podemos estimar:

$$\hat{P}(y=c) = \frac{N_c}{N}$$

donde $N_c$ es el número de imágenes de la clase $c$, y $N$ es el número total de imágenes.

**Segundo término:** $p(x \mid y=c)$ — la **verosimilitud** de la imagen bajo la clase $c$.
Responde: *si la clase fuera $c$, ¿qué tan compatible es esta imagen con lo que normalmente
parecen las imágenes de esa clase?*

**Tercer término:** $p(x)$ — la probabilidad marginal de observar la imagen $x$, sin importar su clase:

$$p(x) = \sum_{k=0}^{9} P(y=k)\;p(x \mid y=k)$$

Pero para clasificar **no necesitamos calcularlo**. ¿Por qué? Porque cuando comparo clases,
la imagen $x$ ya está fija. El denominador $p(x)$ es el mismo para todas las clases:

$$\arg\max_c\, P(y=c \mid x) = \arg\max_c\, \frac{P(y=c)\;p(x \mid y=c)}{p(x)}$$

Como $p(x)$ no depende de $c$, puedo quitarlo:

$$\hat{y} = \arg\max_c\, P(y=c)\;p(x \mid y=c)$$

No estamos diciendo que $p(x)$ no exista. Estamos diciendo que **no afecta qué clase gana**.

---

### 9. ¿Qué nos falta?

Ya tenemos:

$$\hat{y} = \arg\max_c\, P(y=c)\;p(x \mid y=c)$$

El prior $P(y=c)$ es fácil de estimar contando clases.
Pero el término difícil es $p(x \mid y=c)$.
Necesitamos responder: *¿cómo se distribuyen las imágenes dentro de cada clase?*

Aquí hacemos una **suposición de modelado**.
No es una verdad absoluta. Es una decisión que tomamos para poder construir un modelo.

---

### 10. La suposición principal

Suponemos que, dentro de cada clase, las imágenes se distribuyen aproximadamente como una
normal multivariada:

$$x \mid y=c \;\sim\; \mathcal{N}(\mu_c,\, \Sigma_c)$$

Cuidado: no decimos que $y$ sigue una normal. La clase $y$ es discreta ($y \in \{0,1,\dots,9\}$).
Lo que modelamos como normal es la imagen $x$ cuando ya sabemos la clase.

---

### 11. ¿Qué significa $\mu_c$?

La media $\mu_c$ representa la **imagen promedio** de la clase $c$.
Por ejemplo, $\mu_0$ sería el «cero promedio», $\mu_1$ el «uno promedio», $\mu_7$ el «siete promedio».

Como cada imagen tiene 784 píxeles, entonces $\mu_c \in \mathbb{R}^{784}$.
Cada componente de $\mu_c$ representa el valor promedio de un píxel para esa clase.

---

### 12. ¿Qué significa $\Sigma_c$?

La matriz $\Sigma_c$ es la **matriz de covarianza** de la clase $c$.
Como las imágenes viven en $\mathbb{R}^{784}$, entonces $\Sigma_c \in \mathbb{R}^{784 \times 784}$.

- La diagonal contiene varianzas $\Sigma_{c,jj}$, que dicen cuánto cambia el píxel $j$ dentro de la clase $c$.
- Los elementos fuera de la diagonal contienen covarianzas $\Sigma_{c,ij}$, que dicen cómo se
  relacionan dos píxeles diferentes.

Por ejemplo, en el dígito 1, varios píxeles verticales tienden a activarse juntos.
Eso genera correlaciones entre píxeles.

---

### 13. La densidad normal multivariada

Si $x \mid y=c \sim \mathcal{N}(\mu_c, \Sigma_c)$, entonces:

$$p(x \mid y=c)
= \frac{1}{(2\pi)^{d/2}\,|\Sigma_c|^{1/2}}
\exp\!\left(-\frac{1}{2}(x-\mu_c)^{\mathsf{T}}
\Sigma_c^{-1}(x-\mu_c)\right)$$

donde $d = 784$. Esta fórmula parece grande, pero tiene una idea simple:

> Una imagen es más probable bajo una clase si está cerca de la imagen promedio de esa clase,
> teniendo en cuenta cómo varía esa clase.

El término central es $(x - \mu_c)^{\mathsf{T}}\Sigma_c^{-1}(x - \mu_c)$.
Esto mide distancia entre $x$ y $\mu_c$, pero no es distancia euclidiana normal.
Es una distancia que toma en cuenta la varianza y las correlaciones de los píxeles.
Se llama **distancia de Mahalanobis**.

---

### 14. Pero todavía no conocemos $\mu_c$ ni $\Sigma_c$

Hasta ahora dijimos $x \mid y=c \sim \mathcal{N}(\mu_c, \Sigma_c)$, pero esos parámetros no vienen dados.
No sabemos cuál es $\mu_0, \mu_1, \dots, \mu_9$ ni $\Sigma_0, \Sigma_1, \dots, \Sigma_9$.
Entonces debemos estimarlos desde los datos. Ahí entra el **entrenamiento**.

---

### 15. Separación fundamental: entrenar vs. predecir

**Durante entrenamiento**, tengo muchas imágenes con su clase real.
Uso esos datos para aprender los parámetros del modelo: $\hat{\mu}_c$ y $\hat{\Sigma}_c$.

**Durante predicción**, llega una imagen nueva, y uso los parámetros aprendidos para decidir su clase: $\hat{y}$.

Son dos problemas distintos.

---

### 16. Estimación por máxima verosimilitud

Para una clase fija $c$, tengo datos $\mathcal{D}_c = \{x_i : y_i = c\}$.
Supuse que esos datos vienen de $\mathcal{N}(\mu_c, \Sigma_c)$.
La densidad de un dato individual es $p(x_i \mid y_i=c;\, \mu_c, \Sigma_c)$.

Asumimos que las imágenes son **independientes** entre sí, dado el modelo.
Bajo independencia, la probabilidad conjunta de observar varios datos se obtiene multiplicando
sus densidades. Esta función se llama **verosimilitud**:

$$L(\mu_c, \Sigma_c) = \prod_{i:\,y_i=c} p(x_i \mid y_i=c;\, \mu_c, \Sigma_c)$$

No es una probabilidad sobre los parámetros. Es una función de los parámetros.
Los datos ya están fijos; lo que varía es $\mu_c, \Sigma_c$.

Preguntamos: *¿para qué valores de $\mu_c$ y $\Sigma_c$ estos datos serían más verosímiles?*

$$(\hat{\mu}_c,\, \hat{\Sigma}_c) = \arg\max_{\mu_c,\, \Sigma_c} L(\mu_c, \Sigma_c)$$

---

### 17. ¿Por qué usamos logaritmos?

La verosimilitud tiene productos: $L(\theta) = \prod_i p(x_i \mid \theta)$.
Multiplicar muchos números pequeños puede ser incómodo y numéricamente inestable.
Los logaritmos convierten productos en sumas:

$$\ell(\theta) = \log L(\theta) = \sum_i \log p(x_i \mid \theta)$$

El logaritmo es estrictamente creciente: si $a > b$, entonces $\log a > \log b$.
Por tanto, el valor de $\theta$ que maximiza $L(\theta)$ también maximiza $\log L(\theta)$:

$$\arg\max_\theta\, L(\theta) = \arg\max_\theta\, \log L(\theta)$$

El logaritmo cambia la escala, pero no cambia el orden.

---

### 18. Derivación de $\hat{\mu}_c$ por máxima verosimilitud

La log-verosimilitud para la clase $c$ es:

$$\ell(\mu_c, \Sigma_c) = \sum_{i:\,y_i=c}
\left[ -\frac{d}{2}\log(2\pi)
- \frac{1}{2}\log|\Sigma_c|
- \frac{1}{2}(x_i - \mu_c)^{\mathsf{T}} \Sigma_c^{-1} (x_i - \mu_c)
\right]$$

Queremos encontrar el $\mu_c$ que maximiza esta expresión.
Los dos primeros términos no dependen de $\mu_c$, así que solo nos importa el tercero.

Equivalentemente, queremos **minimizar** la suma de distancias de Mahalanobis
(porque hay un signo negativo):

$$\hat{\mu}_c = \arg\min_{\mu_c}
\sum_{i:\,y_i=c} (x_i - \mu_c)^{\mathsf{T}} \Sigma_c^{-1} (x_i - \mu_c)$$

Para encontrar el mínimo, derivamos respecto a $\mu_c$ e igualamos a cero.

Usamos la identidad del cálculo matricial: si $a$ es vector y $A$ es simétrica,
$\frac{\partial}{\partial a} a^{\mathsf{T}} A\, a = 2A\,a$.
Aplicando esto a cada término de la suma:

$$\frac{\partial}{\partial \mu_c}
\sum_{i:\,y_i=c} (x_i - \mu_c)^{\mathsf{T}} \Sigma_c^{-1} (x_i - \mu_c)
= \sum_{i:\,y_i=c} (-2)\, \Sigma_c^{-1} (x_i - \mu_c)$$

Igualando a cero:

$$\sum_{i:\,y_i=c} \Sigma_c^{-1} (x_i - \hat{\mu}_c) = 0$$

Como $\Sigma_c^{-1}$ es invertible, podemos multiplicar por $\Sigma_c$ por la izquierda:

$$\sum_{i:\,y_i=c} (x_i - \hat{\mu}_c) = 0$$

Separando la suma:

$$\sum_{i:\,y_i=c} x_i - N_c\, \hat{\mu}_c = 0$$

Despejando:

$$\boxed{\hat{\mu}_c = \frac{1}{N_c} \sum_{i:\,y_i=c} x_i}$$

Esto significa: *tomo todas las imágenes cuya etiqueta real es $c$, las sumo píxel a píxel,
y divido por cuántas hay.* Si $c=3$, entonces $\hat{\mu}_3$ es el promedio de todos los tres.

---

### 19. Derivación de $\hat{\Sigma}_c$ por máxima verosimilitud

Ahora derivamos $\ell$ respecto a $\Sigma_c$.
Sustituimos $\hat{\mu}_c$ (ya encontrado) en la log-verosimilitud.

Primero, reescribimos la suma de distancias de Mahalanobis usando la traza.
Para cualquier vector $v$ y matriz $A$: $v^{\mathsf{T}} A\, v = \operatorname{tr}(A\, v v^{\mathsf{T}})$.
Entonces:

$$\sum_{i:\,y_i=c} (x_i - \hat{\mu}_c)^{\mathsf{T}} \Sigma_c^{-1} (x_i - \hat{\mu}_c)
= \sum_{i:\,y_i=c} \operatorname{tr}\!\bigl(\Sigma_c^{-1} (x_i - \hat{\mu}_c)(x_i - \hat{\mu}_c)^{\mathsf{T}}\bigr)$$

Usando la linealidad de la traza:

$$= \operatorname{tr}\!\left(\Sigma_c^{-1}
\underbrace{\sum_{i:\,y_i=c} (x_i - \hat{\mu}_c)(x_i - \hat{\mu}_c)^{\mathsf{T}}}_{S_c}\right)
= \operatorname{tr}(\Sigma_c^{-1} S_c)$$

donde definimos la **matriz de dispersión** de la clase $c$:

$$S_c = \sum_{i:\,y_i=c} (x_i - \hat{\mu}_c)(x_i - \hat{\mu}_c)^{\mathsf{T}}$$

Ahora la log-verosimilitud (ignorando constantes que no dependen de $\Sigma_c$) es:

$$\ell(\Sigma_c) = -\frac{N_c}{2} \log|\Sigma_c| - \frac{1}{2} \operatorname{tr}(\Sigma_c^{-1} S_c) + \text{const}$$

Para maximizar respecto a $\Sigma_c$, usamos dos identidades del cálculo matricial
(para $\Sigma_c$ simétrica definida positiva):

$$\frac{\partial}{\partial \Sigma_c} \log|\Sigma_c| = \Sigma_c^{-1},
\qquad
\frac{\partial}{\partial \Sigma_c} \operatorname{tr}(\Sigma_c^{-1} S_c) = -\Sigma_c^{-1} S_c\, \Sigma_c^{-1}$$

Aplicando ambas:

$$\frac{\partial \ell}{\partial \Sigma_c}
= -\frac{N_c}{2}\, \Sigma_c^{-1} + \frac{1}{2}\, \Sigma_c^{-1} S_c\, \Sigma_c^{-1}$$

Igualando a cero:

$$-\frac{N_c}{2}\, \Sigma_c^{-1} + \frac{1}{2}\, \Sigma_c^{-1} S_c\, \Sigma_c^{-1} = 0$$

Multiplicamos por $\Sigma_c$ por la izquierda y por la derecha:

$$-\frac{N_c}{2}\, \mathbf{I}_d + \frac{1}{2}\, S_c\, \Sigma_c^{-1} = 0$$

Multiplicamos por la izquierda por $\Sigma_c$:

$$-\frac{N_c}{2}\, \Sigma_c + \frac{1}{2}\, S_c = 0$$

Despejando:

$$\boxed{\hat{\Sigma}_c = \frac{1}{N_c} \sum_{i:\,y_i=c} (x_i - \hat{\mu}_c)(x_i - \hat{\mu}_c)^{\mathsf{T}}}$$

Esto significa:
1. Tomo una imagen $x_i$ de la clase $c$.
2. Calculo cuánto se aleja del promedio: $x_i - \hat{\mu}_c$.
3. Construyo el producto externo: $(x_i - \hat{\mu}_c)(x_i - \hat{\mu}_c)^{\mathsf{T}}$.
4. Promedio eso sobre todas las imágenes de la clase.

Así obtengo una matriz que describe cómo varían los píxeles dentro de esa clase.

---

### 20. Aplicando logaritmo a la regla de decisión

Recordemos que la regla de clasificación era:

$$\hat{y} = \arg\max_c\, P(y=c)\;p(x \mid y=c)$$

Aplicamos logaritmo. Como $\log(ab) = \log a + \log b$:

$$\hat{y} = \arg\max_c \bigl[\log P(y=c) + \log p(x \mid y=c)\bigr]$$

---

### 21. Sustituimos la normal multivariada

Sabemos que para una normal multivariada:

$$\log p(x \mid y=c)
= -\frac{d}{2}\log(2\pi)
- \frac{1}{2}\log|\Sigma_c|
- \frac{1}{2}(x-\mu_c)^{\mathsf{T}}\Sigma_c^{-1}(x-\mu_c)$$

Entonces $\log P(y=c) + \log p(x \mid y=c)$ se convierte en:

$$\log P(y=c)
- \frac{d}{2}\log(2\pi)
- \frac{1}{2}\log|\Sigma_c|
- \frac{1}{2}(x-\mu_c)^{\mathsf{T}}\Sigma_c^{-1}(x-\mu_c)$$

Pero $-\frac{d}{2}\log(2\pi)$ es igual para todas las clases. Como no depende de $c$,
se puede eliminar del $\arg\max$.

---

### 22. La regla de clasificación final

$$\boxed{\hat{y} = \arg\max_c \left[
\log \hat{P}(y=c)
- \frac{1}{2}\log|\hat{\Sigma}_c|
- \frac{1}{2}(x-\hat{\mu}_c)^{\mathsf{T}}\hat{\Sigma}_c^{-1}(x-\hat{\mu}_c)
\right]}$$

Cada término tiene interpretación:
- $\log \hat{P}(y=c)$: premia clases más frecuentes.
- $-\frac{1}{2}\log|\hat{\Sigma}_c|$: penaliza clases con mucha dispersión.
  El determinante $|\hat{\Sigma}_c|$ mide, de cierta manera, el «volumen» de la nube gaussiana.
- $-\frac{1}{2}(x-\hat{\mu}_c)^{\mathsf{T}}\hat{\Sigma}_c^{-1}(x-\hat{\mu}_c)$: mide qué tan lejos está $x$
  de la media de la clase (distancia de Mahalanobis).

Entonces una clase gana si:
1. era razonablemente probable antes de ver la imagen;
2. su distribución no es demasiado dispersa;
3. la imagen está cerca de su media según la geometría de esa clase.

---

### 23. Regularización

Como $d = 784$ es grande, $\hat{\Sigma}_c$ puede estar mal condicionada o ser singular.
Además necesitamos calcular $\hat{\Sigma}_c^{-1}$ y $|\hat{\Sigma}_c|$, lo que puede ser difícil numéricamente.

Para garantizar invertibilidad, añadimos una perturbación diagonal:

$$\hat{\Sigma}_c^{\text{reg}} = \hat{\Sigma}_c + \varepsilon\, \mathbf{I}_d, \qquad \varepsilon = 10^{-4}$$

---

### 24. Resumen de implementación

1. **Normalizar** intensidades de píxeles a $[0,1]$.
2. Para cada clase $c = 0, \dots, 9$:
   - $\hat{P}(y=c) \leftarrow N_c / N$
   - $\hat{\mu}_c \leftarrow$ media muestral (MLE)
   - $\hat{\Sigma}_c \leftarrow$ covarianza muestral $+\, 10^{-4}\mathbf{I}$ (MLE, regularizada)
   - Pre‑computar $\hat{\Sigma}_c^{-1}$ y $\log|\hat{\Sigma}_c|$
3. Para cada imagen de prueba $x$:
   - Calcular el puntaje $S_c(x)$ para cada $c$
   - Predecir $\hat{y} = \arg\max_c S_c(x)$

---

### 25. La cadena lógica completa

> Tengo imágenes $x \in \mathbb{R}^{784}$ y clases $y \in \{0, \dots, 9\}$.
> Quiero $f(x) = \hat{y}$.
> Como hay diez clases, escojo la más probable: $\hat{y} = \arg\max_c\, P(y=c \mid x)$.
> Por Bayes: $P(y=c \mid x) = P(y=c)\,p(x \mid y=c) / p(x)$.
> Como $p(x)$ no depende de $c$: $\hat{y} = \arg\max_c\, P(y=c)\,p(x \mid y=c)$.
> Aplicando logaritmo: $\hat{y} = \arg\max_c\,[\log P(y=c) + \log p(x \mid y=c)]$.
> Asumimos $x \mid y=c \sim \mathcal{N}(\mu_c, \Sigma_c)$.
> Estimamos por MLE: $\hat{P}(y=c) = N_c/N$, $\hat{\mu}_c = \frac{1}{N_c}\sum_{i:y_i=c} x_i$,
> $\hat{\Sigma}_c = \frac{1}{N_c}\sum_{i:y_i=c} (x_i - \hat{\mu}_c)(x_i - \hat{\mu}_c)^{\mathsf{T}}$.
> Sustituyendo la densidad normal: obtenemos la regla final.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='liac-arff')

X = X.astype(np.float32)
y = y.astype(np.int64)

print(f'Dimensiones de las imágenes: {X.shape}')
print(f'Dimensiones de las etiquetas: {y.shape}')
print(f'Rango de píxeles: [{X.min()}, {X.max()}]')
print(f'Etiquetas: {np.unique(y)}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'Etiqueta: {y[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
unique, counts = np.unique(y, return_counts=True)
plt.bar(unique, counts)
plt.xlabel('Dígito')
plt.ylabel('Cantidad')
plt.title('Distribución de etiquetas')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000, random_state=42, stratify=y)

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba:         {X_test.shape[0]} muestras')

In [ ]:
# --------------------------------------------------------------------
# 1. Normalizar los datos: cada imagen en R^784 con píxeles en [0,1]
# --------------------------------------------------------------------
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0

print(f'Rango píxeles entrenamiento: [{X_train_norm.min():.3f}, {X_train_norm.max():.3f}]')
print(f'Rango píxeles prueba:         [{X_test_norm.min():.3f},  {X_test_norm.max():.3f}]')

In [ ]:
# --------------------------------------------------------------------
# 2. Estimación de máxima verosimilitud de los parámetros
#    Ajustar una gaussiana multivariada condicional a cada dígito 0–9
#    p(x | y=c) = N(x ; μ_c , Σ_c)
# --------------------------------------------------------------------

classes    = np.unique(y_train)
n_classes  = len(classes)
n_features = X_train_norm.shape[1]          # 784

reg = 1e-4   # regularizar Σ para evitar singularidad

means   = np.zeros((n_classes, n_features))
covs    = np.zeros((n_classes, n_features, n_features))
priors  = np.zeros(n_classes)

for i, c in enumerate(classes):
    X_c        = X_train_norm[y_train == c]
    means[i]   = X_c.mean(axis=0)
    covs[i]    = np.cov(X_c, rowvar=False) + reg * np.eye(n_features)
    priors[i]  = len(X_c) / len(y_train)

print(f'Dimensión de medias:   {means.shape}')
print(f'Dimensión de covarianzas: {covs.shape}')
print(f'Priors:        {priors.round(4)}')
print(f'Suma de priors: {priors.sum():.4f}')

In [ ]:
# --------------------------------------------------------------------
# 3. Pre‑computar Σ_c^{-1} y log|Σ_c|, luego definir el predictor MAP
# --------------------------------------------------------------------

cov_invs      = np.zeros_like(covs)
log_det_covs  = np.zeros(n_classes)

for i in range(n_classes):
    cov_invs[i]                = np.linalg.inv(covs[i])
    _, log_det_covs[i]         = np.linalg.slogdet(covs[i])


def predict(X, means, cov_invs, log_det_covs, priors):
    """Clasificar cada muestra usando la regla MAP.
    Maximizar  log p(x | μ_c, Σ_c) + log P(c)  sobre las clases c.
    """
    n_samples    = X.shape[0]
    log_posterior = np.zeros((n_samples, n_classes))

    for i in range(n_classes):
        diff        = X - means[i]          # (n_samples, n_features)
        mahalanobis = np.sum(diff @ cov_invs[i] * diff, axis=1)

        log_posterior[:, i] = (
            -0.5 * (n_features * np.log(2 * np.pi) + log_det_covs[i] + mahalanobis)
            + np.log(priors[i])
        )

    return np.argmax(log_posterior, axis=1), log_posterior


y_pred, log_post = predict(X_test_norm, means, cov_invs, log_det_covs, priors)
print(f'Dimensiones de predicciones: {y_pred.shape}')

In [ ]:
# --------------------------------------------------------------------
# 4. Evaluación
# --------------------------------------------------------------------

accuracy = np.mean(y_pred == y_test)
print(f'Exactitud en prueba: {accuracy:.4f}  ({accuracy*100:.2f} %)')

print('\nExactitud por clase:')
for c in classes:
    mask = y_test == c
    if mask.sum() > 0:
        acc_c = np.mean(y_pred[mask] == c)
        print(f'  Dígito {c}: {acc_c:.4f}  ({mask.sum()} muestras)')

In [ ]:
# --------------------------------------------------------------------
# 5. Matrices de covarianza por clase
# --------------------------------------------------------------------

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, c in enumerate(classes):
    ax = axes.flat[i]
    # Mostrar la submatriz 28x28 de la covarianza (píxeles en la misma posición)
    cov_diag = np.diag(covs[i]).reshape(28, 28)
    im = ax.imshow(cov_diag, cmap='hot', interpolation='nearest')
    ax.set_title(f'Dígito {c}', fontsize=11)
    ax.axis('off')

fig.suptitle('Varianza por píxel de cada clase (diagonal de $\\hat{\\Sigma}_c$)', fontsize=13)
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Varianza')
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------------------------
# 6. Animación — inferencias correctas (con media de clase y confianza)
# --------------------------------------------------------------------
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

correct_mask   = y_pred == y_test
correct_idx    = np.where(correct_mask)[0]

n_show = min(80, len(correct_idx))
show   = correct_idx[:n_show]

# Calcular confianzas (softmax sobre log_post)
from scipy.special import softmax
confidences_all = np.max(softmax(log_post, axis=1), axis=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4),
                         gridspec_kw={'width_ratios': [1, 1, 1]})

ax_img  = axes[0]
ax_mean = axes[1]
ax_bar  = axes[2]

img_disp  = ax_img.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
mean_disp = ax_mean.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
ax_img.set_title('Imagen de prueba')
ax_mean.set_title('Media de la clase predicha')
ax_img.axis('off')
ax_mean.axis('off')

# Barras de probabilidad posterior
bar_positions = np.arange(n_classes)
bars = ax_bar.bar(bar_positions, np.zeros(n_classes), color='steelblue')
ax_bar.set_xticks(bar_positions)
ax_bar.set_xlabel('Clase')
ax_bar.set_ylabel('P(y=c | x)')
ax_bar.set_ylim(0, 1.05)
ax_bar.set_title('Probabilidad posterior')

conf_text = fig.suptitle('', fontsize=12, y=0.98)

def update(frame):
    i = show[frame]
    img_disp.set_data(X_test_norm[i].reshape(28, 28))
    pred_class = y_pred[i]
    mean_disp.set_data(means[pred_class].reshape(28, 28))
    probs = softmax(log_post[i:i+1], axis=1)[0]
    for b, p in zip(bars, probs):
        b.set_height(p)
        b.set_color('crimson' if b.get_x() == pred_class else 'steelblue')
    conf = confidences_all[i]
    conf_text.set_text(f'Verdadero: {y_test[i]}   |   Predicho: {pred_class}   |   Confianza: {conf:.2%}')
    return img_disp, mean_disp, conf_text

ani = FuncAnimation(fig, update, frames=n_show, interval=300, blit=False, repeat=True)
plt.tight_layout()
plt.close()
HTML(ani.to_jshtml())

In [ ]:
# --------------------------------------------------------------------
# 7. Animación — errores de predicción (con media de clase y confianza)
# --------------------------------------------------------------------

error_mask = y_pred != y_test
error_idx  = np.where(error_mask)[0]

n_show = min(80, len(error_idx))
show   = error_idx[:n_show]

fig, axes = plt.subplots(1, 3, figsize=(12, 4),
                         gridspec_kw={'width_ratios': [1, 1, 1]})

ax_img  = axes[0]
ax_mean = axes[1]
ax_bar  = axes[2]

img_disp  = ax_img.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
mean_disp = ax_mean.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
ax_img.set_title('Imagen de prueba')
ax_mean.set_title('Media de la clase predicha')
ax_img.axis('off')
ax_mean.axis('off')

bar_positions = np.arange(n_classes)
bars = ax_bar.bar(bar_positions, np.zeros(n_classes), color='steelblue')
ax_bar.set_xticks(bar_positions)
ax_bar.set_xlabel('Clase')
ax_bar.set_ylabel('P(y=c | x)')
ax_bar.set_ylim(0, 1.05)
ax_bar.set_title('Probabilidad posterior')

conf_text = fig.suptitle('', fontsize=12, color='red', fontweight='bold', y=0.98)

def update(frame):
    i = show[frame]
    img_disp.set_data(X_test_norm[i].reshape(28, 28))
    pred_class = y_pred[i]
    mean_disp.set_data(means[pred_class].reshape(28, 28))
    probs = softmax(log_post[i:i+1], axis=1)[0]
    for b, p in zip(bars, probs):
        b.set_height(p)
        b.set_color('crimson' if b.get_x() == pred_class else 'steelblue')
    conf = confidences_all[i]
    conf_text.set_text(f'Verdadero: {y_test[i]}   |   Predicho (ERROR): {pred_class}   |   Confianza: {conf:.2%}')
    return img_disp, mean_disp, conf_text

ani = FuncAnimation(fig, update, frames=n_show, interval=400, blit=False, repeat=True)
plt.tight_layout()
plt.close()
HTML(ani.to_jshtml())